# AlpasFarm: Train YOLOv8 on 4skwhnrscr Goat Anatomical & Health Dataset

This Google Colab notebook automates end-to-end training of **YOLOv8** on the **4skwhnrscr-2** goat dataset (2,991 images, 15,072 annotations across `goat_face`, `eye`, `mouth`, `ear`, and `goat_body`).

### Detected Anatomical & Health Targets:
- **Class 0 (`goat_face`)**: Facial landmark detection & head symmetry
- **Class 1 (`eye`)**: Ocular screening (FAMACHA conjunctival pallor, discharge, opacity)
- **Class 2 (`mouth`)**: Muzzle inspection (Contagious Ecthyma / Orf scabs, nasal discharge)
- **Class 3 (`ear`)**: Ear posture (drooping lethargy indicator, mange mite lesions)
- **Class 4 (`goat_body`)**: Full body posture, BCS (1-5), and bloat detection

### Step 1: Install Dependencies & Check GPU Acceleration

In [ ]:
!pip install -q ultralytics torchvision pillow matplotlib pyyaml
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Note: Go to Runtime -> Change runtime type -> Select T4 GPU for 10x faster training!")

### Step 2: Load & Prepare 4skwhnrscr Dataset
You can either **mount Google Drive** or **upload `dataset_1.tar.xz` and `dataset_2.tar.xz`** from your computer.

In [ ]:
import os, tarfile, shutil, random
from pathlib import Path

# Check if dataset archives exist in current folder or Google Drive
ds1_tar = Path('/content/dataset_1.tar.xz')
ds2_tar = Path('/content/dataset_2.tar.xz')

# If using Google Drive (optional)
drive_ds1 = Path('/content/drive/MyDrive/4skwhnrscr-2/dataset_1.tar.xz')
drive_ds2 = Path('/content/drive/MyDrive/4skwhnrscr-2/dataset_2.tar.xz')
if drive_ds1.exists():
    ds1_tar = drive_ds1
if drive_ds2.exists():
    ds2_tar = drive_ds2

RAW_DS1 = Path('/content/raw_1')
RAW_DS2 = Path('/content/raw_2')
OUTPUT_DIR = Path('/content/yolo_dataset')

if ds1_tar.exists():
    print(f"Extracting {ds1_tar}...")
    with tarfile.open(ds1_tar, 'r:*') as tar:
        tar.extractall(RAW_DS1)

if ds2_tar.exists():
    print(f"Extracting {ds2_tar}...")
    with tarfile.open(ds2_tar, 'r:*') as tar:
        tar.extractall(RAW_DS2)

# If archives not found, prompt for upload
if not RAW_DS1.exists() and not ds1_tar.exists():
    print("Please upload dataset_1.tar.xz from your computer:")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if 'dataset_1' in fn:
            with tarfile.open(fn, 'r:*') as tar:
                tar.extractall(RAW_DS1)
        elif 'dataset_2' in fn:
            with tarfile.open(fn, 'r:*') as tar:
                tar.extractall(RAW_DS2)

print("Dataset extraction complete!")

### Step 3: Format & Partition Dataset (Train / Val / Test Split)

In [ ]:
CLASSES = {
    0: 'goat_face',
    1: 'eye',
    2: 'mouth',
    3: 'ear',
    4: 'goat_body'
}

CLASS_NAME_MAP = {
    '0': 0, 'face': 0, 'goat_face': 0,
    '1': 1, 'eye': 1, 'eyes': 1,
    '2': 2, 'mouth': 2, 'muzzle': 2, 'nose': 2,
    '3': 3, 'ear': 3, 'ears': 3,
    '4': 4, 'goat': 4, 'body': 4, 'goat_body': 4
}

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ['train', 'val', 'test']:
    (OUTPUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

samples = []
img_extensions = {'.jpg', '.jpeg', '.png', '.webp'}

# 1. Scan raw_1
if RAW_DS1.exists():
    raw1_files = list(RAW_DS1.rglob('*'))
    img_files = [f for f in raw1_files if f.suffix.lower() in img_extensions]
    print(f"Found {len(img_files)} images in dataset_1...")
    for img_path in img_files:
        txt_path = img_path.with_suffix('.txt')
        if not txt_path.exists():
            continue
        try:
            with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
                lines = [l.strip() for l in f.readlines() if l.strip()]
            cleaned = []
            for line in lines:
                parts = line.split()
                if len(parts) >= 5:
                    c = parts[0].lower()
                    if c in CLASS_NAME_MAP:
                        cid = CLASS_NAME_MAP[c]
                        cx, cy, w, h = map(float, parts[1:5])
                        cleaned.append(f"{cid} {max(0.0,min(1.0,cx)):.6f} {max(0.0,min(1.0,cy)):.6f} {max(0.001,min(1.0,w)):.6f} {max(0.001,min(1.0,h)):.6f}")
            if cleaned:
                samples.append({'img_path': img_path, 'labels': cleaned})
        except Exception:
            continue

# 2. Scan raw_2
if RAW_DS2.exists():
    raw2_imgs = [f for f in RAW_DS2.rglob('*') if f.suffix.lower() in img_extensions]
    print(f"Found {len(raw2_imgs)} images in dataset_2...")
    for img_path in raw2_imgs:
        samples.append({'img_path': img_path, 'labels': ['4 0.500000 0.500000 0.850000 0.850000']})

print(f"Total unified samples: {len(samples)}")

# 3. Split 80/10/10
random.seed(42)
random.shuffle(samples)
total = len(samples)
train_end = int(0.80 * total)
val_end = int(0.90 * total)

splits = {
    'train': samples[:train_end],
    'val': samples[train_end:val_end],
    'test': samples[val_end:]
}

for split_name, split_list in splits.items():
    print(f"Writing {len(split_list)} samples to {split_name}...")
    for idx, sample in enumerate(split_list):
        dest_img = OUTPUT_DIR / 'images' / split_name / f"goat_{split_name}_{idx:05d}{sample['img_path'].suffix.lower()}"
        dest_txt = OUTPUT_DIR / 'labels' / split_name / f"goat_{split_name}_{idx:05d}.txt"
        shutil.copy2(sample['img_path'], dest_img)
        with open(dest_txt, 'w', encoding='utf-8') as f:
            f.write('\n'.join(sample['labels']) + '\n')

# 4. Generate Linux data.yaml
data_yaml_content = f'''# AlpasFarm 4skwhnrscr Dataset
path: {OUTPUT_DIR.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

nc: {len(CLASSES)}
names: {list(CLASSES.values())}
'''
data_yaml_path = Path('/content/data.yaml')
with open(data_yaml_path, 'w', encoding='utf-8') as f:
    f.write(data_yaml_content)

print("Generated /content/data.yaml:")
print(data_yaml_content)

### Step 4: Train YOLOv8 on Cloud GPU

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 model
model = YOLO('yolov8n.pt')

# Train model for 50 epochs
results = model.train(
    data='/content/data.yaml',
    epochs=50,
    batch=16,
    imgsz=640,
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=15,
    save=True,
    plots=True,
    project='/content/runs',
    name='4skwhnrscr_goat_model',
    exist_ok=True
)

### Step 5: Evaluate Model Performance & Metrics

In [ ]:
from IPython.display import Image, display

# Validate model
metrics = model.val(data='/content/data.yaml', split='val')
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

# Display training results and curves
results_img = Path('/content/runs/4skwhnrscr_goat_model/results.png')
if results_img.exists():
    display(Image(filename=str(results_img)))

### Step 6: Export & Download Trained Weights (best.pt / best.onnx)

In [ ]:
# Export model to ONNX
onnx_path = model.export(format='onnx', imgsz=640)
print(f"ONNX Model exported at: {onnx_path}")

from google.colab import files
best_pt = Path('/content/runs/4skwhnrscr_goat_model/weights/best.pt')
if best_pt.exists():
    print("Downloading best.pt...")
    files.download(str(best_pt))

if Path(onnx_path).exists():
    print("Downloading best.onnx...")
    files.download(str(onnx_path))